# Ordered Logistic Regression Results Dataset Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2 Croissant dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # this is an object, not a dict

print(f"Dataset Name: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")
print(f"Identifier: {getattr(metadata, 'identifier', None)}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}\n")

## 2. Data Overview
Review available *record sets*, *fields*, and their `@id` values.

`mlcroissant` exposes record sets via the `dataset.record_sets` property. Each record set defines a logical table (analogous to a DataFrame), with each field representing a column, identified by its unique `@id`.

In [ ]:
# List all available record sets and their fields (all by @id)

if not dataset.record_sets:
    print("No record sets found in the Croissant schema.")
else:
    print("Available Record Sets and Fields:")
    for recset in dataset.record_sets:
        recset_id = getattr(recset, '@id', None)
        recset_name = getattr(recset, 'name', None)
        print(f"- Record Set: {recset_name} (@id: {recset_id})")
        # List fields for this record set
        if hasattr(recset, 'fields') and recset.fields:
            for field in recset.fields:
                field_id = getattr(field, '@id', None)
                field_name = getattr(field, 'name', None)
                print(f"    - Field: {field_name} (@id: {field_id})")
        else:
            print("    (No fields defined)")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, using record set and field `@id`s from the overview.

> **Note:** If no record set exists in the schema, this section will gracefully indicate that no data can be loaded.

In [ ]:
# Extract data from each record set (@id referenced everywhere)

import warnings

dataframes = {}

if not dataset.record_sets:
    warnings.warn("No record sets are defined in the Croissant schema—data loading skipped.")
else:
    record_set_ids = [getattr(recset, '@id') for recset in dataset.record_sets]
    print(f"Found record set IDs: {record_set_ids}")
    for recset in dataset.record_sets:
        recset_id = getattr(recset, '@id')
        print(f"\nLoading records for record set '@id': {recset_id}")
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Fields (@id) in this DataFrame: {list(df.columns)}")
        print(df.head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate standard EDA and preprocessing: filter, normalize, group, always referencing columns and fields by their `@id`.

> If no record sets were defined above, this section is skipped.

In [ ]:
# Example EDA: filter, normalize, group

if not dataframes:
    print("No dataframes loaded (no record sets in schema) — skipping EDA.")
else:
    # Pick the first available record set for illustration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using record set @id: {record_set_id}")
    
    # Try to find the first numeric field
    numeric_field_id = None
    if not df.empty:
        for col in df.columns:
            # Try converting column to numeric
            try:
                _ = pd.to_numeric(df[col], errors='coerce')
                # Choose the first successful one
                if df[col].dropna().apply(lambda x: isinstance(x, (int, float)) or str(x).replace('.','',1).isdigit()).any():
                    numeric_field_id = col
                    break
            except Exception:
                continue
    
    if numeric_field_id is None:
        print("No numeric fields found in this record set.")
    else:
        # Ensure numeric dtype
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())
        
        # Try to use a grouping field (first non-numeric field)
        group_field_id = None
        for col in filtered_df.columns:
            if col != numeric_field_id and filtered_df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")

## 5. Visualization
Visualize distributions or relationships (as feasible for loaded data and available columns/fields).

In [ ]:
import matplotlib.pyplot as plt

if not dataframes:
    print("No dataframes loaded — skipping visualization.")
else:
    # Use the selected record set and numeric field from the EDA step above
    try:
        rec_id = record_set_id
        df = dataframes[rec_id]
        if numeric_field_id and numeric_field_id in df.columns:
            plt.figure(figsize=(8,5))
            df[numeric_field_id].hist(bins=20)
            plt.xlabel(numeric_field_id)
            plt.ylabel('Count')
            plt.title(f'Distribution of "{numeric_field_id}" (@id)')
            plt.show()
        else:
            print("No numeric field detected for plotting.")
    except Exception as e:
        print("Visualization not possible (data or columns missing):", e)

## 6. Conclusion
In this notebook, you learned how to load and explore a dataset defined by a Croissant schema using the `mlcroissant` library. You:
- Inspected available record sets and fields (all referenced by `@id`).
- Loaded record data into pandas DataFrames.
- Carried out sample EDA and normalization using only `@id` references for entities.
- Visualized distributions for numeric fields when available.

When working with Croissant datasets, always refer to record sets, fields, and columns by their `@id` for clarity, traceability, and reproducibility.